In [1]:
import sys
print(sys.executable)

c:\Users\hunda\anaconda3\envs\hospital_stats\python.exe


# Hospital Readmission and Quality Analytics - Statistical Analysis

**Author:** Abhishek Hundalekar  
**Project:** CMS Hospital Quality Portfolio Project  
**Stage:** 5 of 6 - Statistical Analysis  
**Data Source:** PostgreSQL `hospital_analytics.v_hospital_master` (5,432 hospitals)

## Business Question
Which hospital characteristics and quality measures are statistically associated with higher readmission rates and lower star ratings?

## Statistical Tests Performed
1. Descriptive statistics and distribution checks
2. Correlation analysis
3. Chi-square test of independence
4. Two-sample group comparison (t-test)
5. One-way ANOVA with Tukey HSD post-hoc
6. Cohen's d and eta-squared effect sizes
7. 95 percent confidence intervals

**Note:** All findings are observed associations from public CMS data, not causal claims.

In [2]:
# Cell 2: Imports and setup
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from sqlalchemy import create_engine
from dotenv import load_dotenv

# Load environment variables from .env in parent folder
load_dotenv(dotenv_path='../.env')

# Display settings
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)
sns.set_style('whitegrid')

print("All libraries loaded successfully")

All libraries loaded successfully


In [3]:
# Diagnostic: check library versions and memory
import sys
import psutil

print(f"Python: {sys.version}")
print(f"Executable: {sys.executable}")
print(f"Available RAM: {psutil.virtual_memory().available / 1e9:.2f} GB")
print(f"Total RAM: {psutil.virtual_memory().total / 1e9:.2f} GB")

import pandas, numpy, scipy, matplotlib, seaborn
print(f"\npandas: {pandas.__version__}")
print(f"numpy: {numpy.__version__}")
print(f"matplotlib: {matplotlib.__version__}")
print(f"seaborn: {seaborn.__version__}")

Python: 3.11.15 | packaged by conda-forge | (main, Jun 11 2026, 03:27:10) [MSC v.1944 64 bit (AMD64)]
Executable: c:\Users\hunda\anaconda3\envs\hospital_stats\python.exe
Available RAM: 4.02 GB
Total RAM: 16.84 GB

pandas: 3.0.3
numpy: 2.4.6
matplotlib: 3.11.0
seaborn: 0.13.2


In [4]:
# Cell 3: Connect to PostgreSQL and load v_hospital_master
db_config = {
    'host': os.getenv('DB_HOST'),
    'port': os.getenv('DB_PORT'),
    'database': os.getenv('DB_NAME'),
    'user': os.getenv('DB_USER'),
    'password': os.getenv('DB_PASSWORD')
}

connection_string = (
    f"postgresql+psycopg2://{db_config['user']}:{db_config['password']}"
    f"@{db_config['host']}:{db_config['port']}/{db_config['database']}"
)

engine = create_engine(connection_string)

# Load the master view
df = pd.read_sql("SELECT * FROM v_hospital_master", engine)

print(f"Loaded {len(df):,} hospitals with {df.shape[1]} columns")
df.head()

Loaded 5,432 hospitals with 31 columns


,facility_id,facility_name,city_town,state,zip_code,hospital_type,hospital_ownership,emergency_services,hospital_overall_rating,avg_err,min_err,max_err,total_discharges,total_readmissions,measures_with_err,readmission_tier,outcome_total_measures,mortality_measures,complication_measures,better_count,worse_count,same_count,outcome_status,patient_star_rating,patient_min_star,patient_max_star,avg_linear_score,completed_surveys,response_rate_percent,patient_experience_tier,data_coverage
0,010001,SOUTHEAST HEALTH MEDICAL CENTER,DOTHAN,AL,36301,Acute Care Hospitals,Government - Hospital District or Authority,Yes,4.0,0.9784,0.9370,1.0233,1692.0,280.0,6.0,Average,20.0,6.0,13.0,0.0,1.0,19.0,Below Average,3.67,3.0,5.0,87.25,952.0,17.0,Good,Complete
1,010005,MARSHALL MEDICAL CENTERS,BOAZ,AL,35957,Acute Care Hospitals,Government - Hospital District or Authority,Yes,3.0,0.9181,0.8602,1.0087,588.0,82.0,4.0,Excellent,20.0,6.0,13.0,0.0,1.0,17.0,Below Average,3.11,2.0,4.0,84.75,705.0,17.0,Good,Complete
2,010006,NORTH ALABAMA MEDICAL CENTER,FLORENCE,AL,35630,Acute Care Hospitals,Proprietary,Yes,2.0,1.0002,0.9195,1.0527,1599.0,259.0,6.0,Average,20.0,6.0,13.0,0.0,4.0,16.0,Below Average,2.11,1.0,4.0,82.00,1814.0,18.0,Average,Complete
3,010007,MIZELL MEMORIAL HOSPITAL,OPP,AL,36467,Acute Care Hospitals,Voluntary non-profit - Private,Yes,1.0,1.0399,1.0232,1.0620,123.0,28.0,3.0,Average,20.0,6.0,13.0,0.0,1.0,12.0,Below Average,NaN,NaN,NaN,NaN,97.0,17.0,No Data,Complete
4,010011,ST. VINCENT'S EAST,BIRMINGHAM,AL,35235,Acute Care Hospitals,Voluntary non-profit - Private,Yes,3.0,1.0275,0.9697,1.1213,787.0,136.0,5.0,Average,20.0,6.0,13.0,0.0,0.0,20.0,Average,2.78,2.0,3.0,84.63,887.0,21.0,Average,Complete


## Test 1: Descriptive Statistics and Distribution Checks

Before running inferential tests, examine data shape, missing values, and distributions of key numeric variables.

In [10]:
# Cell 5: Data types and missing values
info_df = pd.DataFrame({
    'dtype': df.dtypes,
    'missing_count': df.isnull().sum(),
    'missing_percent': (df.isnull().sum() / len(df) * 100).round(1),
    'unique_values': df.nunique()
})
info_df

,dtype,missing_count,missing_percent,unique_values
facility_id,str,0,0.0,5432
facility_name,str,0,0.0,5301
city_town,str,0,0.0,3048
state,str,0,0.0,56
zip_code,int64,0,0.0,4724
hospital_type,str,0,0.0,8
hospital_ownership,str,0,0.0,12
emergency_services,str,0,0.0,2
hospital_overall_rating,float64,2250,41.4,5
avg_err,float64,2607,48.0,1429


In [6]:
# Cell 6: Summary statistics for key numeric variables
key_numeric = ['hospital_overall_rating', 'avg_err', 'patient_star_rating', 
               'avg_linear_score', 'response_rate_percent', 
               'total_discharges', 'completed_surveys']

df[key_numeric].describe().round(2)

,hospital_overall_rating,avg_err,patient_star_rating,avg_linear_score,response_rate_percent,total_discharges,completed_surveys
count,3182.00,2825.00,3176.00,3176.00,3956.00,2564.00,3956.00
mean,3.21,1.00,3.27,85.90,22.95,909.67,582.02
std,1.09,0.06,0.82,3.40,8.36,920.68,764.94
min,1.00,0.55,1.00,66.50,3.00,0.00,25.00
25%,2.00,0.97,2.78,83.88,17.00,276.00,137.00
50%,3.00,1.00,3.22,86.00,22.00,647.00,397.50
75%,4.00,1.03,3.89,88.13,27.00,1260.50,669.25
max,5.00,1.31,5.00,97.50,74.00,11138.00,11713.00


In [7]:
# Cell 7 (lighter): Distribution plots - one at a time
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend, more stable
import matplotlib.pyplot as plt

plot_vars = ['hospital_overall_rating', 'avg_err', 'patient_star_rating', 
             'avg_linear_score', 'response_rate_percent', 'total_discharges']

for col in plot_vars:
    data = df[col].dropna()
    plt.figure(figsize=(8, 4))
    plt.hist(data, bins=30, edgecolor='black', alpha=0.7, color='steelblue')
    plt.axvline(data.mean(), color='red', linestyle='--', label=f'Mean: {data.mean():.2f}')
    plt.axvline(data.median(), color='green', linestyle='--', label=f'Median: {data.median():.2f}')
    plt.title(f'{col} (n={len(data):,})')
    plt.legend()
    plt.tight_layout()
    plt.savefig(f'../docs/dist_{col}.png', dpi=80, bbox_inches='tight')
    plt.close()

print("All 6 distribution plots saved to docs folder")

All 6 distribution plots saved to docs folder


In [8]:
# Cell 8: Shapiro-Wilk normality test on key variables
# Note: Shapiro is unreliable for n > 5000, so we use a 5000-row sample
normality_results = []

for col in ['avg_err', 'patient_star_rating', 'avg_linear_score', 'response_rate_percent']:
    data = df[col].dropna()
    sample = data.sample(min(5000, len(data)), random_state=42)
    stat, p = stats.shapiro(sample)
    normality_results.append({
        'variable': col,
        'n_used': len(sample),
        'shapiro_stat': round(stat, 4),
        'p_value': f"{p:.2e}",
        'is_normal_alpha_0.05': 'Yes' if p > 0.05 else 'No'
    })

pd.DataFrame(normality_results)

,variable,n_used,shapiro_stat,p_value,is_normal_alpha_0.05
0,avg_err,2825,0.8943,2.29e-40,No
1,patient_star_rating,3176,0.9909,2.46e-13,No
2,avg_linear_score,3176,0.9888,3.70e-15,No
3,response_rate_percent,3956,0.9566,8.60e-33,No


## Test 2: Correlation Analysis

Measure strength and direction of associations between key hospital quality metrics.

**Method:** Pearson (linear) and Spearman (rank-based, more robust to non-normality).
**Interpretation guide:**
- |r| < 0.3: weak
- 0.3 to 0.5: moderate
- 0.5 to 0.7: strong
- > 0.7: very strong

In [11]:
# Cell 10: Correlation matrix - Pearson and Spearman
corr_vars = ['hospital_overall_rating', 'avg_err', 'patient_star_rating', 
             'avg_linear_score', 'response_rate_percent', 
             'total_discharges', 'completed_surveys']

corr_df = df[corr_vars].dropna()
print(f"Complete-case sample size: {len(corr_df):,} hospitals\n")

pearson_corr = corr_df.corr(method='pearson').round(3)
spearman_corr = corr_df.corr(method='spearman').round(3)

print("PEARSON CORRELATION")
print(pearson_corr)
print("\nSPEARMAN CORRELATION")
print(spearman_corr)

Complete-case sample size: 2,408 hospitals

PEARSON CORRELATION
                         hospital_overall_rating  avg_err  patient_star_rating  avg_linear_score  response_rate_percent  total_discharges  completed_surveys
hospital_overall_rating                    1.000   -0.449                0.524             0.522                  0.297             0.153              0.141
avg_err                                   -0.449    1.000               -0.234            -0.235                 -0.213            -0.018             -0.032
patient_star_rating                        0.524   -0.234                1.000             0.974                  0.420            -0.058              0.029
avg_linear_score                           0.522   -0.235                0.974             1.000                  0.403            -0.048              0.041
response_rate_percent                      0.297   -0.213                0.420             0.403                  1.000             0.117              

In [13]:
# Cell 11: Heatmap visualization of Spearman correlations
plt.figure(figsize=(10, 7))
sns.heatmap(spearman_corr, annot=True, cmap='RdBu_r', center=0, 
            vmin=-1, vmax=1, fmt='.2f', square=True, linewidths=0.5,
            cbar_kws={'label': 'Spearman correlation'})
plt.title('Correlation Matrix: Hospital Quality Metrics (Spearman)', fontsize=12)
plt.tight_layout()
plt.savefig('../docs/correlation_heatmap.png', dpi=100, bbox_inches='tight')
plt.close()
print("Heatmap saved to docs/correlation_heatmap.png")

Heatmap saved to docs/correlation_heatmap.png


In [ ]:
# Cell 12a Statistical significance of key correlations
from scipy.stats import spearmanr

key_pairs = [
    ('hospital_overall_rating', 'patient_star_rating'),
    ('hospital_overall_rating', 'avg_err'),
    ('patient_star_rating', 'avg_err'),
    ('response_rate_percent', 'patient_star_rating'),
    ('total_discharges', 'avg_err')
]

results = []
for var1, var2 in key_pairs:
    pair_df = df[[var1, var2]].dropna()
    rho, p = spearmanr(pair_df[var1], pair_df[var2])
    results.append({
        'variable_1': var1,
        'variable_2': var2,
        'n': len(pair_df),
        'spearman_rho': round(rho, 3),
        'p_value': f"{p:.2e}",
        'significant_alpha_0.05': 'Yes' if p < 0.05 else 'No',
        'strength': 'strong' if abs(rho) > 0.5 else 'moderate' if abs(rho) > 0.3 else 'weak'
    })

pd.DataFrame(results)

,variable_1,variable_2,n,spearman_rho,p_value,significant_alpha_0.05,strength
0,hospital_overall_rating,patient_star_rating,2929,0.479,6.67e-168,Yes,moderate
1,hospital_overall_rating,avg_err,2628,-0.441,8.88e-126,Yes,moderate
2,patient_star_rating,avg_err,2622,-0.256,2.13e-40,Yes,weak
3,response_rate_percent,patient_star_rating,3176,0.499,1.18e-199,Yes,moderate
4,total_discharges,avg_err,2525,0.010,6.29e-01,No,weak


In [14]:
# Cell 12b: Drop redundant variable due to near-perfect correlation with patient_star_rating
df = df.drop(columns=['avg_linear_score'])
print(f"Dropped 'avg_linear_score'. New shape: {df.shape}")

Dropped 'avg_linear_score'. New shape: (5432, 30)


## Test 3: Chi-Square Test of Independence

**Question:** Is hospital overall rating independent of ownership type?

**Hypotheses:**
- H0 (null): Rating and ownership are independent (no association)
- H1 (alternative): Rating and ownership are associated

**Method:** Chi-square test of independence + Cramer's V effect size

In [15]:
# Cell 14: Build contingency table of ownership vs overall rating
chi_df = df[['hospital_ownership', 'hospital_overall_rating']].dropna()
print(f"Sample size: {len(chi_df):,} hospitals\n")

contingency = pd.crosstab(chi_df['hospital_ownership'], 
                          chi_df['hospital_overall_rating'],
                          margins=True, margins_name='Total')
contingency

Sample size: 3,182 hospitals



hospital_overall_rating,1.0,2.0,3.0,4.0,5.0,Total
hospital_ownership,,,,,,
Government - Federal,0,5,4,3,0,12
Government - Hospital District or Authority,20,63,80,51,16,230
Government - Local,20,46,54,50,11,181
Government - State,5,10,10,8,4,37
Physician,0,3,7,4,5,19
Proprietary,51,167,144,116,26,504
Tribal,0,0,1,0,1,2
Veterans Health Administration,0,7,18,37,50,112
Voluntary non-profit - Church,11,26,78,84,23,222


In [16]:
# Cell 15: Chi-square test of independence
from scipy.stats import chi2_contingency

contingency_matrix = pd.crosstab(chi_df['hospital_ownership'], 
                                  chi_df['hospital_overall_rating'])

chi2, p_value, dof, expected = chi2_contingency(contingency_matrix)

# Cramer's V effect size
n = contingency_matrix.sum().sum()
min_dim = min(contingency_matrix.shape) - 1
cramers_v = np.sqrt(chi2 / (n * min_dim))

results = {
    'Chi-square statistic': round(chi2, 2),
    'Degrees of freedom': dof,
    'p-value': f"{p_value:.2e}",
    'Sample size': n,
    "Cramer's V": round(cramers_v, 3),
    'Effect size interpretation': (
        'small' if cramers_v < 0.2 else 
        'moderate' if cramers_v < 0.4 else 
        'strong'
    ),
    'Reject H0 at alpha=0.05': 'Yes' if p_value < 0.05 else 'No'
}

pd.DataFrame([results]).T.rename(columns={0: 'Value'})

,Value
Chi-square statistic,290.22
Degrees of freedom,40
p-value,1.06e-39
Sample size,3182
Cramer's V,0.151
Effect size interpretation,small
Reject H0 at alpha=0.05,Yes


In [17]:
# Cell 16: Chi-square assumption check
# Rule: at least 80% of expected frequencies should be >= 5
expected_df = pd.DataFrame(expected, 
                            index=contingency_matrix.index,
                            columns=contingency_matrix.columns).round(1)

cells_below_5 = (expected < 5).sum()
total_cells = expected.size
percent_below_5 = round(cells_below_5 / total_cells * 100, 1)

print(f"Expected frequencies below 5: {cells_below_5} of {total_cells} cells ({percent_below_5} percent)")
print(f"Assumption met (should be under 20 percent): {'Yes' if percent_below_5 < 20 else 'No'}")
print("\nExpected frequency table:")
expected_df

Expected frequencies below 5: 15 of 55 cells (27.3 percent)
Assumption met (should be under 20 percent): No

Expected frequency table:


hospital_overall_rating,1.0,2.0,3.0,4.0,5.0
hospital_ownership,,,,,
Government - Federal,0.8,2.5,3.7,3.6,1.4
Government - Hospital District or Authority,14.4,47.9,71.3,68.7,27.8
Government - Local,11.3,37.7,56.1,54.0,21.8
Government - State,2.3,7.7,11.5,11.0,4.5
Physician,1.2,4.0,5.9,5.7,2.3
Proprietary,31.5,104.9,156.3,150.5,60.8
Tribal,0.1,0.4,0.6,0.6,0.2
Veterans Health Administration,7.0,23.3,34.7,33.4,13.5
Voluntary non-profit - Church,13.9,46.2,68.9,66.3,26.8


In [18]:
# Cell 17: Row percentages to see rating distribution within each ownership type
row_pct = pd.crosstab(chi_df['hospital_ownership'], 
                      chi_df['hospital_overall_rating'],
                      normalize='index') * 100

row_pct = row_pct.round(1)
row_pct['n_hospitals'] = chi_df.groupby('hospital_ownership').size()
row_pct.sort_values('n_hospitals', ascending=False)

hospital_overall_rating,1.0,2.0,3.0,4.0,5.0,n_hospitals
hospital_ownership,,,,,,
Voluntary non-profit - Private,4.9,18.1,31.2,32.7,13.2,1611
Proprietary,10.1,33.1,28.6,23.0,5.2,504
Voluntary non-profit - Other,5.2,17.5,34.9,28.2,14.3,252
Government - Hospital District or Authority,8.7,27.4,34.8,22.2,7.0,230
Voluntary non-profit - Church,5.0,11.7,35.1,37.8,10.4,222
Government - Local,11.0,25.4,29.8,27.6,6.1,181
Veterans Health Administration,0.0,6.2,16.1,33.0,44.6,112
Government - State,13.5,27.0,27.0,21.6,10.8,37
Physician,0.0,15.8,36.8,21.1,26.3,19


In [19]:
# Cell 18: Fix assumption violation by grouping small ownership categories
chi_df2 = chi_df.copy()

# Group small categories into 'Other'
small_categories = ['Tribal', 'Physician', 'Government - Federal', 'Government - State']
chi_df2['ownership_grouped'] = chi_df2['hospital_ownership'].replace(
    {cat: 'Other/Small' for cat in small_categories}
)

# Rerun chi-square
contingency2 = pd.crosstab(chi_df2['ownership_grouped'], 
                            chi_df2['hospital_overall_rating'])

chi2_2, p_2, dof_2, expected_2 = chi2_contingency(contingency2)
cramers_v_2 = np.sqrt(chi2_2 / (contingency2.sum().sum() * (min(contingency2.shape) - 1)))
cells_below_5_2 = (expected_2 < 5).sum()
percent_below_5_2 = round(cells_below_5_2 / expected_2.size * 100, 1)

print("REGROUPED CHI-SQUARE RESULTS")
results2 = {
    'Chi-square statistic': round(chi2_2, 2),
    'p-value': f"{p_2:.2e}",
    "Cramer's V": round(cramers_v_2, 3),
    'Cells below 5 (percent)': percent_below_5_2,
    'Assumption met': 'Yes' if percent_below_5_2 < 20 else 'No',
    'Conclusion': 'Rating depends on ownership' if p_2 < 0.05 else 'Independent'
}
pd.DataFrame([results2]).T.rename(columns={0: 'Value'})

REGROUPED CHI-SQUARE RESULTS


,Value
Chi-square statistic,274.3
p-value,2.95e-42
Cramer's V,0.147
Cells below 5 (percent),2.5
Assumption met,Yes
Conclusion,Rating depends on ownership


## Test 4: Two-Sample Group Comparison

**Question:** Do Proprietary (for-profit) and Voluntary non-profit - Private hospitals differ in average Excess Readmission Ratio (ERR)?

**Hypotheses:**
- H0 (null): Mean ERR is the same in both groups
- H1 (alternative): Mean ERR differs between groups

**Method:** Welch's t-test (primary) + Mann-Whitney U (robustness) + Cohen's d (effect size)

In [21]:
# Cell 20: Extract ERR data for the two ownership groups
group_a_name = 'Proprietary'
group_b_name = 'Voluntary non-profit - Private'

group_a = df[(df['hospital_ownership'] == group_a_name) & 
             (df['avg_err'].notna())]['avg_err']
group_b = df[(df['hospital_ownership'] == group_b_name) & 
             (df['avg_err'].notna())]['avg_err']

summary = pd.DataFrame({
    'group': [group_a_name, group_b_name],
    'n': [len(group_a), len(group_b)],
    'mean_err': [round(group_a.mean(), 4), round(group_b.mean(), 4)],
    'std_err': [round(group_a.std(), 4), round(group_b.std(), 4)],
    'median_err': [round(group_a.median(), 4), round(group_b.median(), 4)],
    'min': [round(group_a.min(), 4), round(group_b.min(), 4)],
    'max': [round(group_a.max(), 4), round(group_b.max(), 4)]
})
summary

,group,n,mean_err,std_err,median_err,min,max
0,Proprietary,547,1.0126,0.0608,1.0118,0.6587,1.3037
1,Voluntary non-profit - Private,1395,0.9986,0.0502,0.9974,0.5939,1.2484


In [22]:
# Cell 21: Run both parametric and non-parametric tests
from scipy.stats import ttest_ind, mannwhitneyu

# Welch's t-test (unequal variances)
t_stat, t_p = ttest_ind(group_a, group_b, equal_var=False)

# Mann-Whitney U (non-parametric)
u_stat, u_p = mannwhitneyu(group_a, group_b, alternative='two-sided')

# Cohen's d effect size (pooled standard deviation)
pooled_std = np.sqrt(((len(group_a) - 1) * group_a.var() + 
                       (len(group_b) - 1) * group_b.var()) / 
                       (len(group_a) + len(group_b) - 2))
cohens_d = (group_a.mean() - group_b.mean()) / pooled_std

# 95% CI for the difference in means (Welch's method)
mean_diff = group_a.mean() - group_b.mean()
se_diff = np.sqrt(group_a.var()/len(group_a) + group_b.var()/len(group_b))
dof_welch = (group_a.var()/len(group_a) + group_b.var()/len(group_b))**2 / (
    (group_a.var()/len(group_a))**2/(len(group_a)-1) + 
    (group_b.var()/len(group_b))**2/(len(group_b)-1)
)
ci_lower = mean_diff - stats.t.ppf(0.975, dof_welch) * se_diff
ci_upper = mean_diff + stats.t.ppf(0.975, dof_welch) * se_diff

results = {
    'Group A': f"{group_a_name} (n={len(group_a)})",
    'Group B': f"{group_b_name} (n={len(group_b)})",
    'Mean difference (A - B)': round(mean_diff, 4),
    '95% CI of difference': f"[{ci_lower:.4f}, {ci_upper:.4f}]",
    "Welch's t-statistic": round(t_stat, 3),
    "Welch's p-value": f"{t_p:.2e}",
    'Mann-Whitney U': round(u_stat, 0),
    'Mann-Whitney p-value': f"{u_p:.2e}",
    "Cohen's d": round(cohens_d, 3),
    'Effect size interpretation': (
        'negligible' if abs(cohens_d) < 0.2 else
        'small' if abs(cohens_d) < 0.5 else
        'medium' if abs(cohens_d) < 0.8 else
        'large'
    ),
    'Reject H0 at alpha=0.05': 'Yes' if t_p < 0.05 else 'No'
}

pd.DataFrame([results]).T.rename(columns={0: 'Value'})

,Value
Group A,Proprietary (n=547)
Group B,Voluntary non-profit - Private (n=1395)
Mean difference (A - B),0.014
95% CI of difference,"[0.0083, 0.0198]"
Welch's t-statistic,4.791
Welch's p-value,1.96e-06
Mann-Whitney U,450894.0
Mann-Whitney p-value,4.37e-10
Cohen's d,0.262
Effect size interpretation,small


In [24]:
# Cell 22: Boxplot comparison
plt.figure(figsize=(9, 5))
data_to_plot = [group_a, group_b]
tick_labels = [f'{group_a_name}\n(n={len(group_a)})', 
               f'{group_b_name}\n(n={len(group_b)})']

bp = plt.boxplot(data_to_plot, tick_labels=tick_labels, patch_artist=True, showmeans=True,
                  meanprops={'marker':'D','markerfacecolor':'red','markersize':8})
bp['boxes'][0].set_facecolor('lightcoral')
bp['boxes'][1].set_facecolor('lightblue')

plt.axhline(y=1.0, color='gray', linestyle='--', alpha=0.6, label='ERR = 1.0 (national avg)')
plt.ylabel('Average Excess Readmission Ratio (ERR)')
plt.title('ERR Distribution: Proprietary vs Voluntary Non-Profit Private')
plt.legend()
plt.tight_layout()
plt.savefig('../docs/boxplot_err_ownership.png', dpi=100, bbox_inches='tight')
plt.close()
print("Boxplot saved to docs/boxplot_err_ownership.png")

Boxplot saved to docs/boxplot_err_ownership.png


## Test 5: One-Way ANOVA with Tukey HSD

**Question:** Do mean ERR values differ across ownership types?

**Hypotheses:**
- H0 (null): All ownership groups have the same mean ERR
- H1 (alternative): At least one group differs

**Method:** One-way ANOVA + eta-squared + Kruskal-Wallis + Tukey HSD post-hoc

In [25]:
# Cell 24: Prepare data for ANOVA - filter to groups with enough hospitals
anova_df = df[['hospital_ownership', 'avg_err']].dropna()

# Keep only ownership types with at least 30 hospitals (statistical reliability)
group_counts = anova_df['hospital_ownership'].value_counts()
valid_groups = group_counts[group_counts >= 30].index.tolist()
anova_df = anova_df[anova_df['hospital_ownership'].isin(valid_groups)]

print(f"Total hospitals used: {len(anova_df):,}")
print(f"Number of ownership groups: {anova_df['hospital_ownership'].nunique()}\n")

group_summary = anova_df.groupby('hospital_ownership')['avg_err'].agg(
    ['count', 'mean', 'std', 'median']
).round(4).sort_values('mean')
group_summary

Total hospitals used: 2,807
Number of ownership groups: 8



,count,mean,std,median
hospital_ownership,,,,
Physician,56,0.9221,0.1658,0.9541
Voluntary non-profit - Church,199,0.9923,0.0472,0.9891
Voluntary non-profit - Other,236,0.9965,0.0512,0.9943
Voluntary non-profit - Private,1395,0.9986,0.0502,0.9974
Government - Hospital District or Authority,208,0.9991,0.0389,0.9969
Government - Local,130,1.0023,0.0395,1.0010
Government - State,36,1.0028,0.0470,1.0014
Proprietary,547,1.0126,0.0608,1.0118


In [26]:
# Cell 25: One-way ANOVA and eta-squared
from scipy.stats import f_oneway, kruskal

# Split into list of arrays by group
groups = [anova_df[anova_df['hospital_ownership'] == g]['avg_err'].values 
          for g in valid_groups]

# One-way ANOVA
f_stat, anova_p = f_oneway(*groups)

# Eta-squared effect size: SS_between / SS_total
grand_mean = anova_df['avg_err'].mean()
ss_between = sum(len(g) * (g.mean() - grand_mean)**2 for g in groups)
ss_total = sum((anova_df['avg_err'] - grand_mean)**2)
eta_squared = ss_between / ss_total

# Kruskal-Wallis (non-parametric robustness)
h_stat, kw_p = kruskal(*groups)

results = {
    'Number of groups': len(valid_groups),
    'Total n': len(anova_df),
    'F-statistic': round(f_stat, 3),
    'ANOVA p-value': f"{anova_p:.2e}",
    'Eta-squared': round(eta_squared, 4),
    'Eta-squared interpretation': (
        'negligible' if eta_squared < 0.01 else
        'small' if eta_squared < 0.06 else
        'medium' if eta_squared < 0.14 else
        'large'
    ),
    'Kruskal-Wallis H': round(h_stat, 3),
    'Kruskal-Wallis p-value': f"{kw_p:.2e}",
    'Reject H0 at alpha=0.05': 'Yes' if anova_p < 0.05 else 'No'
}

pd.DataFrame([results]).T.rename(columns={0: 'Value'})

,Value
Number of groups,8
Total n,2807
F-statistic,20.447
ANOVA p-value,5.83e-27
Eta-squared,0.0486
Eta-squared interpretation,small
Kruskal-Wallis H,75.835
Kruskal-Wallis p-value,9.70e-14
Reject H0 at alpha=0.05,Yes


In [27]:
# Cell 26: Tukey HSD - which specific pairs differ
tukey = pairwise_tukeyhsd(anova_df['avg_err'], 
                          anova_df['hospital_ownership'],
                          alpha=0.05)

# Convert Tukey output to DataFrame for readable display
tukey_df = pd.DataFrame(data=tukey._results_table.data[1:], 
                         columns=tukey._results_table.data[0])

# Show only significant pairs, sorted by effect size
sig_pairs = tukey_df[tukey_df['reject'] == True].copy()
sig_pairs['abs_diff'] = sig_pairs['meandiff'].abs()
sig_pairs = sig_pairs.sort_values('abs_diff', ascending=False)

print(f"Total pairwise comparisons: {len(tukey_df)}")
print(f"Significant pairs (Tukey-adjusted): {len(sig_pairs)}\n")
print("SIGNIFICANT PAIRS (sorted by effect magnitude):")
sig_pairs[['group1', 'group2', 'meandiff', 'p-adj', 'lower', 'upper']].round(4)

Total pairwise comparisons: 28
Significant pairs (Tukey-adjusted): 10

SIGNIFICANT PAIRS (sorted by effect magnitude):


,group1,group2,meandiff,p-adj,lower,upper
18,Physician,Proprietary,0.0905,0.0000,0.0668,0.1143
13,Government - State,Physician,-0.0808,0.0000,-0.1169,-0.0446
8,Government - Local,Physician,-0.0802,0.0000,-0.1072,-0.0531
2,Government - Hospital District or Authority,Physician,-0.0770,0.0000,-0.1024,-0.0515
21,Physician,Voluntary non-profit - Private,0.0765,0.0000,0.0535,0.0996
20,Physician,Voluntary non-profit - Other,0.0744,0.0000,0.0493,0.0996
19,Physician,Voluntary non-profit - Church,0.0702,0.0000,0.0446,0.0958
22,Proprietary,Voluntary non-profit - Church,-0.0204,0.0003,-0.0344,-0.0064
23,Proprietary,Voluntary non-profit - Other,-0.0161,0.0052,-0.0293,-0.0029
24,Proprietary,Voluntary non-profit - Private,-0.0140,0.0000,-0.0226,-0.0055


In [28]:
# Cell 27: Boxplot across all ownership groups
plt.figure(figsize=(12, 6))
sorted_groups = group_summary.sort_values('mean').index.tolist()
data_to_plot = [anova_df[anova_df['hospital_ownership'] == g]['avg_err'].values 
                for g in sorted_groups]

bp = plt.boxplot(data_to_plot, tick_labels=sorted_groups, patch_artist=True, 
                  showmeans=True,
                  meanprops={'marker':'D','markerfacecolor':'red','markersize':6})
for box in bp['boxes']:
    box.set_facecolor('lightsteelblue')

plt.axhline(y=1.0, color='gray', linestyle='--', alpha=0.6, label='ERR = 1.0 (national avg)')
plt.xticks(rotation=30, ha='right')
plt.ylabel('Average ERR')
plt.title('ERR Distribution by Ownership Type (sorted by mean)')
plt.legend()
plt.tight_layout()
plt.savefig('../docs/boxplot_err_all_ownership.png', dpi=100, bbox_inches='tight')
plt.close()
print("Boxplot saved to docs/boxplot_err_all_ownership.png")

Boxplot saved to docs/boxplot_err_all_ownership.png


## Test 6: 95 Percent Confidence Intervals

Report range estimates for key metrics to convey uncertainty alongside point estimates.

In [29]:
# Cell 29: 95% CI for mean ERR by ownership type
def mean_ci(data, confidence=0.95):
    n = len(data)
    mean = data.mean()
    se = data.std() / np.sqrt(n)
    margin = stats.t.ppf((1 + confidence) / 2, n - 1) * se
    return mean, mean - margin, mean + margin

ci_results = []
for group in valid_groups:
    data = anova_df[anova_df['hospital_ownership'] == group]['avg_err']
    mean, lower, upper = mean_ci(data)
    ci_results.append({
        'ownership': group,
        'n': len(data),
        'mean_err': round(mean, 4),
        'ci_lower': round(lower, 4),
        'ci_upper': round(upper, 4),
        'ci_width': round(upper - lower, 4)
    })

pd.DataFrame(ci_results).sort_values('mean_err').reset_index(drop=True)

,ownership,n,mean_err,ci_lower,ci_upper,ci_width
0,Physician,56,0.9221,0.8777,0.9665,0.0888
1,Voluntary non-profit - Church,199,0.9923,0.9857,0.9988,0.0132
2,Voluntary non-profit - Other,236,0.9965,0.9899,1.0031,0.0131
3,Voluntary non-profit - Private,1395,0.9986,0.9959,1.0012,0.0053
4,Government - Hospital District or Authority,208,0.9991,0.9937,1.0044,0.0106
5,Government - Local,130,1.0023,0.9954,1.0091,0.0137
6,Government - State,36,1.0028,0.9869,1.0187,0.0318
7,Proprietary,547,1.0126,1.0075,1.0177,0.0102


In [30]:
# Cell 30: 95% CI for proportion of 5-star hospitals by ownership
def proportion_ci(successes, n, confidence=0.95):
    p = successes / n
    z = stats.norm.ppf((1 + confidence) / 2)
    se = np.sqrt(p * (1 - p) / n)
    return p, p - z*se, p + z*se

prop_df = df[['hospital_ownership', 'hospital_overall_rating']].dropna()
top_valid = prop_df['hospital_ownership'].value_counts()
top_valid = top_valid[top_valid >= 30].index.tolist()

prop_results = []
for group in top_valid:
    grp = prop_df[prop_df['hospital_ownership'] == group]
    successes = (grp['hospital_overall_rating'] == 5).sum()
    n = len(grp)
    p, lower, upper = proportion_ci(successes, n)
    prop_results.append({
        'ownership': group,
        'n': n,
        'five_star_count': successes,
        'proportion': round(p, 3),
        'ci_lower': round(max(0, lower), 3),
        'ci_upper': round(min(1, upper), 3)
    })

pd.DataFrame(prop_results).sort_values('proportion', ascending=False).reset_index(drop=True)

,ownership,n,five_star_count,proportion,ci_lower,ci_upper
0,Veterans Health Administration,112,50,0.446,0.354,0.538
1,Voluntary non-profit - Other,252,36,0.143,0.100,0.186
2,Voluntary non-profit - Private,1611,212,0.132,0.115,0.148
3,Government - State,37,4,0.108,0.008,0.208
4,Voluntary non-profit - Church,222,23,0.104,0.064,0.144
5,Government - Hospital District or Authority,230,16,0.070,0.037,0.102
6,Government - Local,181,11,0.061,0.026,0.096
7,Proprietary,504,26,0.052,0.032,0.071


In [31]:
# Cell 31: Forest plot of mean ERR CIs
ci_df = pd.DataFrame(ci_results).sort_values('mean_err')

plt.figure(figsize=(10, 5))
y_pos = range(len(ci_df))
plt.errorbar(ci_df['mean_err'], y_pos,
             xerr=[ci_df['mean_err'] - ci_df['ci_lower'],
                   ci_df['ci_upper'] - ci_df['mean_err']],
             fmt='o', capsize=5, color='steelblue')
plt.yticks(y_pos, ci_df['ownership'])
plt.axvline(x=1.0, color='red', linestyle='--', alpha=0.5, label='National avg (ERR=1.0)')
plt.xlabel('Mean ERR with 95% CI')
plt.title('Mean ERR by Ownership Type (95 percent Confidence Intervals)')
plt.legend()
plt.tight_layout()
plt.savefig('../docs/forest_plot_err_ci.png', dpi=100, bbox_inches='tight')
plt.close()
print("Forest plot saved to docs/forest_plot_err_ci.png")

Forest plot saved to docs/forest_plot_err_ci.png


## Stage 5 Summary: Statistical Analysis Findings

### Tests Performed and Key Results

| Test | Finding | Effect Size |
|---|---|---|
| Normality (Shapiro-Wilk) | All 4 metrics non-normal; CLT protects large-sample tests | n/a |
| Correlation (Spearman) | Rating and ERR moderately linked (rho = -0.44) | Moderate |
| Chi-square | Rating depends on ownership (p < 1e-42) | Cramer's V = 0.15 (small) |
| t-test | Proprietary ERR 0.014 higher than non-profit private | Cohen's d = 0.26 (small) |
| ANOVA + Tukey HSD | 10 of 28 pairs differ significantly | Eta-sq = 0.049 (small) |
| 95% CIs | VA hospitals dominate 5-star share (44.6%, CI [35%, 54%]) | n/a |

### Business Findings (Observed Associations Only)

1. **VA hospitals lead on quality.** 44.6 percent are 5-star, more than 2x any other ownership type, with non-overlapping CIs.
2. **Proprietary hospitals consistently trail.** Highest ERR (1.013), lowest 5-star share (5.2 percent), worst across all pairwise comparisons.
3. **Physician-owned hospitals show lowest ERR but with 3x higher variance.** Likely selection bias (specialty facilities, elective patients).
4. **Hospital size does not predict readmissions.** total_discharges vs avg_err correlation = 0.03.
5. **CMS methodology validated.** Overall rating correlates with patient experience (0.51) and readmissions (-0.45).

### Statistical Discipline Applied

- Reported both parametric and non-parametric tests for robustness
- Reported effect sizes alongside every p-value to prevent overclaiming
- Fixed chi-square assumption failure by regrouping small categories
- Dropped redundant variable (avg_linear_score) after finding r = 0.99 with patient_star_rating
- Framed all findings as observed associations, not causal claims

### Caveats

- Observational data - no causal claims
- Missing data 41-48 percent on rating and ERR (CMS did not rate all hospitals)
- Physician-owned hospital findings likely reflect selection bias

In [32]:
# Cell 32: Consolidated findings table for portfolio and interview reference
findings = pd.DataFrame([
    {'test': 'Shapiro-Wilk (avg_err)', 'statistic': 0.894, 'p_value': '2.29e-40', 
     'effect_size': 'n/a', 'conclusion': 'Non-normal (CLT applies)'},
    {'test': 'Spearman (rating vs ERR)', 'statistic': -0.445, 'p_value': '<1e-100', 
     'effect_size': 'moderate', 'conclusion': 'Higher rating = lower ERR'},
    {'test': 'Spearman (patient vs linear)', 'statistic': 0.986, 'p_value': '<1e-100', 
     'effect_size': 'very strong', 'conclusion': 'Redundant - dropped avg_linear_score'},
    {'test': 'Chi-square (regrouped)', 'statistic': 274.3, 'p_value': '2.95e-42', 
     'effect_size': "Cramer's V = 0.147 (small)", 'conclusion': 'Rating depends on ownership'},
    {'test': "Welch's t-test (Prop vs NP-Private)", 'statistic': 4.791, 'p_value': '1.96e-06', 
     'effect_size': "Cohen's d = 0.26 (small)", 'conclusion': 'Proprietary ERR is 0.014 higher'},
    {'test': 'Mann-Whitney U (robustness)', 'statistic': 450894, 'p_value': '4.37e-10', 
     'effect_size': 'n/a', 'conclusion': 'Confirms t-test result'},
    {'test': 'One-way ANOVA (ERR by ownership)', 'statistic': 20.447, 'p_value': '5.83e-27', 
     'effect_size': 'Eta-sq = 0.049 (small)', 'conclusion': 'Groups differ; 10 of 28 pairs sig'},
    {'test': 'Kruskal-Wallis (robustness)', 'statistic': 75.835, 'p_value': '9.70e-14', 
     'effect_size': 'n/a', 'conclusion': 'Confirms ANOVA result'},
    {'test': '95% CI - VA 5-star proportion', 'statistic': 0.446, 'p_value': 'n/a', 
     'effect_size': 'CI [0.354, 0.538]', 'conclusion': 'VA dominates; non-overlapping CI'},
])

print("STAGE 5 CONSOLIDATED FINDINGS TABLE\n")
findings

STAGE 5 CONSOLIDATED FINDINGS TABLE



,test,statistic,p_value,effect_size,conclusion
0,Shapiro-Wilk (avg_err),0.894,2.29e-40,n/a,Non-normal (CLT applies)
1,Spearman (rating vs ERR),-0.445,<1e-100,moderate,Higher rating = lower ERR
2,Spearman (patient vs linear),0.986,<1e-100,very strong,Redundant - dropped avg_linear_score
3,Chi-square (regrouped),274.300,2.95e-42,Cramer's V = 0.147 (small),Rating depends on ownership
4,Welch's t-test (Prop vs NP-Private),4.791,1.96e-06,Cohen's d = 0.26 (small),Proprietary ERR is 0.014 higher
5,Mann-Whitney U (robustness),450894.000,4.37e-10,n/a,Confirms t-test result
6,One-way ANOVA (ERR by ownership),20.447,5.83e-27,Eta-sq = 0.049 (small),Groups differ; 10 of 28 pairs sig
7,Kruskal-Wallis (robustness),75.835,9.70e-14,n/a,Confirms ANOVA result
8,95% CI - VA 5-star proportion,0.446,n/a,"CI [0.354, 0.538]",VA dominates; non-overlapping CI
